In [ ]:
# ==============================
# 1. LOAD DATA
# ==============================
import pandas as pd

df = pd.read_csv("data/fraudTest.csv")

print(df.shape)
print(df['is_fraud'].value_counts())

# ==============================
# 2. DROP KOLOM TIDAK PENTING
# ==============================
df = df.drop(columns=[
    'Unnamed: 0',
    'first',
    'last',
    'street',
    'trans_num'
])

# ==============================
# 3. FEATURE ENGINEERING (WAKTU & UMUR)
# ==============================
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

df['hour'] = df['trans_date_trans_time'].dt.hour
df['day'] = df['trans_date_trans_time'].dt.day
df['month'] = df['trans_date_trans_time'].dt.month

df['dob'] = pd.to_datetime(df['dob'])
df['age'] = 2025 - df['dob'].dt.year

df = df.drop(columns=['trans_date_trans_time', 'dob'])

# ==============================
# 4. ENCODING KATEGORIKAL
# ==============================
from sklearn.preprocessing import LabelEncoder

cat_cols = df.select_dtypes(include='object').columns

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# ==============================
# 5. PISAH FITUR & TARGET
# ==============================
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

# ==============================
# 6. SPLIT DATA
# ==============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==============================
# 7. TRAIN RANDOM FOREST
# ==============================
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
    max_depth=20
)

rf.fit(X_train, y_train)

# ==============================
# 8. EVALUASI MODEL
# ==============================
from sklearn.metrics import classification_report, confusion_matrix

y_pred = rf.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# ==============================
# 9. SIMPAN MODEL
# ==============================
import joblib
joblib.dump(rf, "rf_fraud_model.pkl")

print("MODEL TERSIMPAN 🔥")


(555719, 23)
is_fraud
0    553574
1      2145
Name: count, dtype: int64
